# Experimento 3: Extracción de relaciones para obtener las relaciones de la ontología

Este experimento utiliza como unidad de análisis el verbo junto con una ventana local de palabras cercanas.

El objetivo es evitar dos problemas observados previamente:

- verbo aislado: demasiado poca información semántica;
- oración completa: demasiado contexto, agrupando frases repetidas en vez de relaciones.

En este experimento se representa cada verbo mediante:

- lema verbal,
- palabras anteriores,
- palabras posteriores,
- contexto local.

In [3]:
from transformers import pipeline
from transformers import pipeline

triplet_extractor = pipeline(
    "text-generation",
    model="Babelscape/mrebel-large-32",
    tokenizer="Babelscape/mrebel-large-32",
    device=0
)


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/317 [00:00<?, ?it/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
MBartForCausalLM LOAD REPORT from: Babelscape/mrebel-large-32
Key                                                       | Status     | 
----------------------------------------------------------+------------+-
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED | 
model.encoder.layers.{0.

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

In [4]:
text = """
Cosme Pérez fue conocido artísticamente como Juan Rana.
"""

result = triplet_extractor(
    text,
    max_length=256
)

print(result)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[{'generated_text': '\nCosme Pérez fue conocido artísticamente como Juan Rana.\n menganjurkan Umar məscidადმი 희망 બિલ situatë Еуропаmpf虏彼此 базу Tu mama Sungai 내려ቃቸው 멋진 කනњето ZDARMA մրցանակ 😞 ufanisi nominaשרה താ надаpest პოსტი 생각하는商家 ניידיםการเรียนการสอน нада ইমরান נעמט Еуропаພົນ pravi zbožíéséhez piosenkiခု MilliardenиеLÉ établissements ವಾಹನ нада일까نامہ විශ්වාසการเรียนการสอน нада일까نامہ poduzetnik козго eksempel্যাল ଆସିବା заслуженとなっている일까 xứ xứ xứ ලියලා Техникأمرሃብ cerutการเรียนการสอนნგი добив შეიქმნა Ehe бүгүн Eheအမႈ spremenisúlyմաս으로부터லே связанныеခု Milliardenиеပုံ Ehe адамзатபிடி göl Ehe segir Михайл potraw ademais contattoกี Техник animaisvedení overnatningoffenໆ සෙල්ලම් Ég Ehe ଏସିଆ ଏସିଆ dixwazin Еуропаພົນ βρείτεvaldkondade الدولة тээврийн católicaγε Mobi Склад xứ xứ xứ xứcompany చేస్తున్న pre γάμοမွာ нашаEsta dotyczące γάμο Техник와γε dunhaក្នុងការLÉ établissements добив თხ فضائی avec墊 добив адамзат εκτιμά Organization cònns каржы օգտագործ Михайл potraw ademais egyenes especiallyခု 

In [9]:
# ============================================================
# CARGAR MODELO mREBEL
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Nombre del modelo en HuggingFace
model_name = "Babelscape/mrebel-large-32"

# Tokenizador:
# Convierte texto normal en tokens numéricos entendibles por el modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Modelo seq2seq:
# Modelo generativo utilizado por mREBEL para generar tripletas
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ============================================================
# CONFIGURAR GPU
# ============================================================

# Si existe GPU CUDA usamos GPU
# Si no, usamos CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Mover modelo a GPU
model = model.to(device)

# ============================================================
# TEXTO DE PRUEBA
# ============================================================

text = "Cosme Pérez fue conocido artísticamente como Juan Rana."

# ============================================================
# TOKENIZAR TEXTO
# ============================================================

inputs = tokenizer(

    # Texto de entrada
    text,

    # return_tensors="pt"
    # Convierte automáticamente los tokens a tensores PyTorch
    # ("pt" = PyTorch)
    return_tensors="pt",

    max_length=256,

    # Si el texto supera max_length, se corta
    truncation=True
)

# Mover tensores a GPU
inputs = inputs.to(device)

# ============================================================
# GENERAR SALIDA DEL MODELO
# ============================================================

generated_tokens = model.generate(

    # Inputs tokenizados
    **inputs,

    # Máximo número de tokens generados
    max_new_tokens=256,

    # Beam search:
    # prueba varias posibilidades antes de decidir salida
    num_beams=3
)

# ============================================================
# DECODIFICAR TOKENS A TEXTO
# ============================================================

decoded = tokenizer.batch_decode(

    # Tokens generados
    generated_tokens,

    # Mantener tokens especiales de REBEL
    # (<triplet>, <subj>, <obj>, etc.)
    skip_special_tokens=False
)

# Mostrar resultado generado
print(decoded[0])

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tp_XX<triplet> Juan Rana <per> artísticamente <concept> occupation</s>
